# 3.3 线性回归的简洁实现

3.2 节手写了数据迭代器、模型、损失函数和优化器。本节改用 PyTorch 提供的高级 API 完成同一个任务，体会深度学习框架如何减少重复代码。

学习目标：

1. 使用 `TensorDataset` 和 `DataLoader` 读取小批量数据；
2. 使用 `nn.Linear` 定义全连接层；
3. 使用 `nn.MSELoss` 定义均方误差损失；
4. 使用 `torch.optim.SGD` 更新参数；
5. 理解训练模式、梯度清零、反向传播和参数更新的标准流程。

本 Notebook 为初学者版本，几乎每一行可执行代码都附有中文注释。

对应教材：[3.3 线性回归的简洁实现](https://zh.d2l.ai/chapter_linear-networks/linear-regression-concise.html)

## 3.3.1 导入工具并固定随机性

`torch` 提供张量运算，`nn` 提供神经网络层和损失函数，`DataLoader` 负责自动打乱、分批和迭代数据。

In [ ]:
import torch  # 导入 PyTorch 主库，用于张量计算和自动微分
from torch import nn  # 从 PyTorch 导入神经网络模块，并使用简称 nn
from torch.utils.data import DataLoader, TensorDataset  # 导入数据集封装器和批量加载器

torch.manual_seed(42)  # 固定随机种子，使参数初始化和数据顺序更容易复现
print("PyTorch version:", torch.__version__)  # 显示当前 PyTorch 版本，便于检查环境

## 3.3.2 生成模拟数据

继续使用 3.2 节的人工线性模型：

$$
y=2x_1-3.4x_2+4.2+\epsilon,
\qquad \epsilon\sim\mathcal N(0,0.01^2).
$$

真实权重和偏置是已知的，因此训练结束后可以直接判断模型是否学对。

In [ ]:
def synthetic_data(w, b, num_examples):  # 定义生成线性回归模拟数据的函数
    X = torch.normal(0, 1, (num_examples, len(w)))  # 从标准正态分布采样特征矩阵 X
    y = X @ w + b  # 根据真实线性关系计算没有噪声的标签，@ 表示矩阵乘法
    y += torch.normal(0, 0.01, y.shape)  # 给标签加入均值为 0、标准差为 0.01 的高斯噪声
    return X, y.reshape(-1, 1)  # 返回特征，并把标签整理成“样本数 × 1”的二维形状

true_w = torch.tensor([2.0, -3.4])  # 设置用于生成数据的真实权重
true_b = 4.2  # 设置用于生成数据的真实偏置
features, labels = synthetic_data(true_w, true_b, 1000)  # 生成 1000 个带噪声样本

print("特征形状:", features.shape)  # 预期输出 torch.Size([1000, 2])
print("标签形状:", labels.shape)  # 预期输出 torch.Size([1000, 1])
print("第一个样本:", features[0], "->", labels[0].item())  # 查看一条具体数据

## 3.3.3 使用框架读取数据

`TensorDataset` 把多个张量按第一维配对。访问第 $i$ 项时，它会同时返回 `features[i]` 和 `labels[i]`。

`DataLoader` 再把数据集包装成可迭代的小批量：

- `batch_size=10`：每批最多包含 10 个样本；
- `shuffle=True`：每个 epoch 开始时重新打乱样本；
- 最后一批不足 10 个样本时仍会正常返回。

In [ ]:
batch_size = 10  # 指定每个小批量包含 10 个样本
dataset = TensorDataset(features, labels)  # 把特征和标签按样本位置配对成一个数据集
data_iter = DataLoader(dataset, batch_size=batch_size, shuffle=True)  # 创建自动打乱并分批的数据加载器

X_batch, y_batch = next(iter(data_iter))  # 从数据加载器中取出第一个小批量
print("批量特征形状:", X_batch.shape)  # 查看批量特征的形状，应为 [10, 2]
print("批量标签形状:", y_batch.shape)  # 查看批量标签的形状，应为 [10, 1]

## 3.3.4 定义模型

`nn.Linear(in_features, out_features)` 实现

$$
\mathbf y=\mathbf X\mathbf W^{\top}+\mathbf b.
$$

本例每个样本有 2 个输入特征，只预测 1 个数，因此使用 `nn.Linear(2, 1)`。

`nn.Sequential` 是一个按顺序执行各层的容器。虽然这里只有一层，仍使用它来展示以后构建多层网络的通用写法。

In [ ]:
net = nn.Sequential(nn.Linear(2, 1))  # 创建一个包含单个线性层的顺序模型：2 个输入、1 个输出
linear_layer = net[0]  # 取得顺序模型中的第 0 层，方便查看和初始化它的参数

print(net)  # 打印网络结构，确认模型由 Linear(2, 1) 构成
print("权重形状:", linear_layer.weight.shape)  # PyTorch 的线性层权重形状为 [输出数, 输入数]
print("偏置形状:", linear_layer.bias.shape)  # 每个输出单元对应一个偏置，因此形状为 [1]

## 3.3.5 初始化模型参数

PyTorch 会自动创建权重和偏置，但也允许手工指定初始化方式。本例让权重服从 $\mathcal N(0,0.01^2)$，并把偏置设为 0，与 3.2 节保持一致。

参数名末尾的下划线（如 `normal_`、`fill_`）表示该操作会直接修改原张量。

In [ ]:
linear_layer.weight.data.normal_(0, 0.01)  # 把权重原地初始化为均值 0、标准差 0.01 的随机数
linear_layer.bias.data.fill_(0)  # 把偏置张量中的所有元素原地设为 0

print("初始权重:", linear_layer.weight.data)  # 查看初始化后的权重数值
print("初始偏置:", linear_layer.bias.data)  # 查看初始化后的偏置数值

## 3.3.6 定义损失函数

`nn.MSELoss()` 默认计算一个批量中所有元素的平均平方误差：

$$
L=\frac{1}{|\mathcal B|}\sum_{i\in\mathcal B}(\hat y_i-y_i)^2.
$$

它与 3.2 节手写的 $\frac12(\hat y-y)^2$ 相差一个常数因子 $2$。这不会改变最优解，但会改变梯度大小；学习率应与损失定义配合。

In [ ]:
loss_fn = nn.MSELoss()  # 创建均方误差损失函数，默认对批量中的损失求平均
example_predictions = net(X_batch)  # 让尚未训练的模型对第一个小批量进行预测
example_loss = loss_fn(example_predictions, y_batch)  # 计算初始预测与真实标签之间的均方误差
print("训练前的小批量损失:", example_loss.item())  # 用 item() 把单元素张量转换成普通 Python 数值

## 3.3.7 定义优化算法

`torch.optim.SGD` 接收需要更新的参数和学习率。`net.parameters()` 会自动提供网络内所有可训练的权重与偏置，因此以后增加更多层时不必逐个列出参数。

In [ ]:
learning_rate = 0.03  # 设置小批量随机梯度下降的学习率
trainer = torch.optim.SGD(net.parameters(), lr=learning_rate)  # 创建 SGD 优化器并交给它管理网络参数
print(trainer)  # 打印优化器配置，确认学习率等设置

## 3.3.8 训练模型

标准 PyTorch 训练循环包含以下步骤：

1. `net.train()`：切换到训练模式；
2. `trainer.zero_grad()`：清空上一批累积的梯度；
3. `net(X)`：前向传播；
4. `loss_fn(...)`：计算损失；
5. `loss.backward()`：反向传播并计算梯度；
6. `trainer.step()`：根据梯度更新参数。

线性层在训练模式和评估模式下行为相同，但养成显式切换模式的习惯很重要，因为 Dropout、批量归一化等层会根据模式改变行为。

In [ ]:
num_epochs = 3  # 指定完整遍历训练集 3 次

for epoch in range(num_epochs):  # 外层循环控制训练轮数
    net.train()  # 把网络切换到训练模式
    for X, y in data_iter:  # 内层循环依次读取当前 epoch 的每个小批量
        trainer.zero_grad()  # 清空上一小批量留下的参数梯度，防止梯度意外累加
        predictions = net(X)  # 前向传播：根据当前参数计算这一批样本的预测值
        batch_loss = loss_fn(predictions, y)  # 计算这一批预测值与真实标签的均方误差
        batch_loss.backward()  # 反向传播：自动计算损失关于所有模型参数的梯度
        trainer.step()  # 根据刚计算出的梯度和学习率更新权重与偏置

    net.eval()  # 把网络切换到评估模式，为整套数据上的评估做准备
    with torch.no_grad():  # 关闭梯度记录，减少评估阶段的内存与计算开销
        epoch_loss = loss_fn(net(features), labels)  # 计算全部训练样本上的平均损失
    print(f"epoch {epoch + 1}, loss {epoch_loss.item():.8f}")  # 输出当前轮数和训练损失

## 3.3.9 检查学到的参数

`nn.Linear` 的权重形状是 `[out_features, in_features]`，本例为 `[1, 2]`，所以用 `.reshape(-1)` 将其整理成长度为 2 的向量，再和真实权重比较。

注意：真实任务通常不知道参数真值，应使用验证集或测试集评价预测能力。

In [ ]:
learned_w = linear_layer.weight.data.reshape(-1)  # 取出训练后的权重并整理成一维向量
learned_b = linear_layer.bias.data.item()  # 取出训练后的单个偏置并转换成 Python 数值
w_error = true_w - learned_w  # 计算每个权重的“真实值减估计值”误差
b_error = true_b - learned_b  # 计算偏置的“真实值减估计值”误差

print("真实权重:", true_w.tolist())  # 显示用于生成数据的真实权重
print("学得权重:", learned_w.tolist())  # 显示模型通过训练估计出的权重
print("权重误差:", w_error.tolist())  # 显示两个权重各自的估计误差
print(f"真实偏置: {true_b:.4f}")  # 显示真实偏置并保留 4 位小数
print(f"学得偏置: {learned_b:.4f}")  # 显示模型估计出的偏置并保留 4 位小数
print(f"偏置误差: {b_error:.6f}")  # 显示偏置误差并保留 6 位小数

assert torch.max(torch.abs(w_error)) < 0.1  # 断言所有权重误差均小于 0.1，用于自动检查训练结果
assert abs(b_error) < 0.1  # 断言偏置误差小于 0.1，用于自动检查训练结果

## 3.3.10 使用模型预测新样本

训练完成后，把新样本整理成 `[样本数, 特征数]` 的二维张量并传给网络，即可得到预测结果。即使只预测一个样本，也要保留批量维度。

In [ ]:
new_X = torch.tensor([[1.5, -2.0]])  # 创建一个新样本，形状为“1 个样本 × 2 个特征”
true_y = new_X @ true_w + true_b  # 使用已知真实参数计算该样本的理论标签

net.eval()  # 把网络切换到评估模式
with torch.no_grad():  # 预测时不需要梯度，因此关闭自动微分记录
    predicted_y = net(new_X)  # 把新样本传入训练好的网络得到预测值

print("理论值:", true_y.item())  # 显示由真实参数计算出的理论值
print("预测值:", predicted_y.item())  # 显示模型根据学得参数给出的预测值

## 3.2 与 3.3 的对应关系

| 训练组成 | 3.2 从零实现 | 3.3 简洁实现 |
|---|---|---|
| 数据集 | 两个普通张量 | `TensorDataset` |
| 小批量读取 | 手写 `data_iter` | `DataLoader` |
| 线性模型 | 手写 `linreg` | `nn.Linear` |
| 参数管理 | 手工创建 `w`、`b` | 层自动注册参数 |
| 平方损失 | 手写 `squared_loss` | `nn.MSELoss` |
| 梯度更新 | 手写 `sgd` | `torch.optim.SGD` |
| 梯度清零 | `param.grad.zero_()` | `trainer.zero_grad()` |

高级 API 没有改变算法本身，只是把通用、容易出错的机械工作封装起来。理解 3.2 节后再学习这些封装，才能知道每个对象在训练循环中承担什么职责。

## 易错点

- **忘记 `zero_grad()`**：PyTorch 默认累加梯度，会让不同小批量的梯度混在一起。
- **标签形状错误**：预测是 `[batch_size, 1]`，标签也应保持相同形状。
- **损失函数用错**：线性回归通常使用 MSE，不应直接套用分类任务的交叉熵。
- **输入特征数不匹配**：`nn.Linear(2, 1)` 要求每个样本最后一维长度为 2。
- **直接比较权重形状**：线性层权重为 `[1, 2]`，真实权重为 `[2]`，比较前应先整理形状。
- **评估时仍记录梯度**：使用 `net.eval()` 和 `torch.no_grad()` 可以避免不必要的开销。
- **认为 `eval()` 会关闭梯度**：它只改变某些层的行为；关闭梯度仍需 `torch.no_grad()`。

## 小结

- PyTorch 可以自动完成数据分批、层定义、参数注册、损失计算和参数更新。
- `nn.Module` 是模型与层的基础，调用模型会执行前向传播。
- `loss.backward()` 负责求梯度，`optimizer.step()` 负责使用梯度更新参数。
- 高级 API 让代码更短，但训练的数学原理与 3.2 节完全一致。
- 一个稳定的训练循环应明确区分训练阶段和评估阶段。

## 练习

1. 把 `nn.Sequential(nn.Linear(2, 1))` 简化成直接使用 `nn.Linear(2, 1)`，相应修改取参数的代码。
2. 将 `batch_size` 改为 1、100 和 1000，比较训练速度与损失变化。
3. 将学习率分别改为 0.003、0.3 和 3.0，观察收敛速度或发散现象。
4. 把 `nn.MSELoss()` 改为 `nn.MSELoss(reduction='sum')`。学习率或训练代码需要怎样调整？
5. 删除 `trainer.zero_grad()` 并重新运行，解释结果为什么异常。
6. 将特征数扩展到 5，相应修改真实权重和 `nn.Linear` 的输入维度。
7. 使用 `torch.optim.Adam` 替换 SGD，比较两者在此简单任务上的结果。
8. 把数据拆分为训练集和测试集，分别计算两部分的 MSE。
9. 尝试不给权重和偏置重新初始化，观察 PyTorch 默认初始化是否仍能收敛。

# 练习参考答案

下面给出一种参考思路。许多实验题没有唯一答案，实际数值会随随机种子、数据和 PyTorch 版本略有变化。建议先自己修改前面的代码观察现象，再对照答案。

## 答案 1：直接使用 `nn.Linear`

`nn.Sequential` 只是层的容器。只有一个线性层时，可以直接把该层当作模型：

```python
net = nn.Linear(2, 1)  # 直接创建线性模型，不再套用 Sequential 容器
net.weight.data.normal_(0, 0.01)  # 把模型权重初始化为均值 0、标准差 0.01 的随机数
net.bias.data.fill_(0)  # 把模型偏置初始化为 0
trainer = torch.optim.SGD(net.parameters(), lr=0.03)  # 创建负责更新该模型参数的 SGD 优化器
learned_w = net.weight.data.reshape(-1)  # 训练后直接从 net.weight 中读取权重
learned_b = net.bias.data.item()  # 训练后直接从 net.bias 中读取偏置
```

训练循环中的 `net(X)` 不需要修改，因为 `nn.Linear` 和 `nn.Sequential` 都是可调用的 `nn.Module`。

## 答案 2：改变批量大小

| `batch_size` | 每个 epoch 的更新次数 | 梯度特点 | 3 个 epoch 时的常见现象 |
|---:|---:|---|---|
| 1 | 1000 | 噪声最大 | 更新频繁，损失有波动，单轮较慢 |
| 10 | 100 | 有适度噪声 | 通常能快速而稳定地收敛 |
| 100 | 10 | 更平滑 | 每轮更新较少，可能需要更多 epoch |
| 1000 | 1 | 完整且稳定 | 3 次参数更新通常不够，需要明显增加 epoch |

批量越大不代表一定收敛越快：它减少了单个 epoch 的更新次数。公平比较时，可以同时记录总更新时间、总参数更新次数和最终损失。

## 答案 3：改变学习率

- `0.003`：每次更新很小，通常稳定但收敛较慢，3 个 epoch 后可能仍有明显误差。
- `0.3`：在这个简单问题上可能快速收敛，也更容易出现损失波动。
- `3.0`：步长通常过大，损失可能迅速增大，最终出现 `inf` 或 `nan`。

若出现 `nan`，说明数值已经溢出。应停止当前实验、重新初始化模型，并降低学习率。不能直接使用已经发散的参数继续训练。

## 答案 4：使用求和损失

`reduction='sum'` 会让梯度约为默认平均损失梯度的 `batch_size` 倍。最稳妥的做法是在反向传播前自行除以当前批量的元素数：

```python
loss_sum_fn = nn.MSELoss(reduction='sum')  # 创建对所有误差直接求和的 MSE 损失
predictions = net(X)  # 使用当前模型计算一个小批量的预测值
batch_loss = loss_sum_fn(predictions, y) / predictions.numel()  # 除以预测元素总数，恢复平均损失尺度
trainer.zero_grad()  # 清空上一批数据留下的梯度
batch_loss.backward()  # 对恢复成平均尺度的损失执行反向传播
trainer.step()  # 使用与原来相同的学习率更新模型参数
```

也可以把学习率近似缩小为原来的 `1 / batch_size`，但最后一批大小可能不同，而且多输出任务的元素数不一定等于样本数，因此显式除以 `predictions.numel()` 更清楚。

## 答案 5：删除 `zero_grad()`

PyTorch 的 `.backward()` 会把新梯度加到已有的 `.grad` 上。删除 `trainer.zero_grad()` 后，第 2 批使用“第 1 批梯度 + 第 2 批梯度”，后续批次还会不断混入已经过时的梯度。结果通常是更新幅度越来越不合理，损失震荡甚至发散。

梯度累加本身也可以被有意用于模拟大批量训练，但那需要在累加固定批数后才执行一次 `step()`，并正确缩放损失；这里直接删除清零操作不属于正确的梯度累加。

## 答案 6：扩展到 5 个特征

数据生成函数本身已使用 `len(w)` 决定特征数，只需更换真实权重并修改线性层输入维度：

```python
true_w_5d = torch.tensor([2.0, -3.4, 1.5, 0.7, -2.2])  # 定义包含 5 个分量的真实权重
true_b_5d = 4.2  # 定义新的真实偏置
features_5d, labels_5d = synthetic_data(true_w_5d, true_b_5d, 1000)  # 生成具有 5 个特征的数据
dataset_5d = TensorDataset(features_5d, labels_5d)  # 把五维特征和标签配对成数据集
data_iter_5d = DataLoader(dataset_5d, batch_size=10, shuffle=True)  # 创建五维数据的小批量加载器
net_5d = nn.Sequential(nn.Linear(5, 1))  # 创建接收 5 个输入特征并输出 1 个数的模型
net_5d[0].weight.data.normal_(0, 0.01)  # 使用小随机数初始化五个权重
net_5d[0].bias.data.fill_(0)  # 把新模型的偏置初始化为 0
trainer_5d = torch.optim.SGD(net_5d.parameters(), lr=0.03)  # 为五维模型创建 SGD 优化器
```

之后把训练循环中的 `net`、`data_iter` 和 `trainer` 分别替换为带 `_5d` 后缀的对象即可。

## 答案 7：使用 Adam

只需替换优化器，训练循环保持不变：

```python
trainer = torch.optim.Adam(net.parameters(), lr=0.01)  # 使用 Adam 管理参数，并采用较常见的较小学习率
```

Adam 会为每个参数维护梯度的一阶和二阶统计量，自适应调整更新尺度。在复杂网络上它经常更容易调参；但本例是简单的凸优化问题，SGD 已能很好地求解，Adam 不一定更快，也不会改变正确的最优解。比较优化器时，应使用相同的数据、初始化和评价标准。

## 答案 8：划分训练集和测试集

下面按前 800 个样本训练、后 200 个样本测试。人工数据原本是独立同分布随机生成的，因此这样切分是可行的；真实的有序数据通常应先随机划分或按业务时间规则划分。

```python
train_features = features[:800]  # 取前 800 个样本的特征作为训练特征
train_labels = labels[:800]  # 取前 800 个样本的标签作为训练标签
test_features = features[800:]  # 取后 200 个样本的特征作为测试特征
test_labels = labels[800:]  # 取后 200 个样本的标签作为测试标签
train_dataset = TensorDataset(train_features, train_labels)  # 把训练特征和训练标签封装成数据集
train_iter = DataLoader(train_dataset, batch_size=10, shuffle=True)  # 创建只读取训练集的数据加载器

net.eval()  # 把训练完成的模型切换到评估模式
with torch.no_grad():  # 关闭梯度记录，因为评价模型不需要反向传播
    train_mse = loss_fn(net(train_features), train_labels)  # 计算训练集上的平均平方误差
    test_mse = loss_fn(net(test_features), test_labels)  # 计算从未参与训练的测试集上的平均平方误差
print("训练集 MSE:", train_mse.item())  # 输出训练集误差
print("测试集 MSE:", test_mse.item())  # 输出测试集误差
```

严格实验时，应在划分数据后重新创建并训练模型，确保测试集从未参与参数更新或超参数选择。

## 答案 9：保留默认初始化

删除手工初始化的两行即可：

```python
net_default = nn.Sequential(nn.Linear(2, 1))  # 创建模型并保留 PyTorch 为 Linear 层提供的默认初始化
trainer_default = torch.optim.SGD(net_default.parameters(), lr=0.03)  # 为默认初始化的模型创建优化器
```

`nn.Linear` 默认不会把参数全部设为 0，而是在与输入维度有关的范围内随机初始化权重和偏置。对于本例的凸线性回归，不同合理初始值通常都能收敛到近似相同的最优参数；初始损失和前几轮的下降轨迹可能不同。深层非凸网络对初始化通常更加敏感。